# BH Curve Models

Analytical models for magnetic B-H relationships in ferromagnetic materials.

**Contents:**
- Fröhlich-Kennelly analytical approximation
- Linear permeability model
- Parameter fitting to measured data

These models are used in motor magnetic circuit analysis and material characterization.

## Executive Summary

**Purpose:** Represent ferromagnetic core saturation in circuit analysis

**What it does:** Model magnetic material B-H curves (linear, Fröhlich saturation).

### Why It Matters
MEC solver needs accurate B-H relationships to compute flux and reluctance in magnetic circuits.

### Quick Start
**Inputs:** Linear permeability μ or fitted Fröhlich parameters (a, b)

**Outputs:** B-H curve models for use in magnetic circuit analysis

## How It Fits Into the Motor Design Workflow

**Upstream (depends on):** Used by 04_material_models.ipynb and 04_mec_solver.ipynb

**Downstream (used by):** See notebook connections

In [1]:
#| hide
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## Theory: Magnetic B-H Relationships

The **B-H curve** describes how magnetic flux density (B, in Tesla) relates to magnetic field intensity (H, in A/m).

### Linear Approximation

For low field strengths, below saturation:

$$B = \mu_0 \cdot \mu_r \cdot H$$

where:
- $\mu_0 = 4\pi \times 10^{-7}$ H/m (permeability of free space)
- $\mu_r$ (relative permeability)

Valid for small H where the material stays in its linear region.

### Fröhlich-Kennelly Model

A better analytical fit for moderate field strengths:

$$B = \frac{H}{a + b |H|}$$

where $a$ and $b$ are fitted parameters.

**References:**
- Fröhlich (1881) — Original investigations
- Hanselman (2003) §3.2 — Motor design applications

In [2]:
#| export
def linear_region(H: np.ndarray, mu_r: float) -> np.ndarray:
    r"""
    Linear BH approximation: B = μ₀ · μr · H
    
    Valid for low flux densities well below saturation.
    
    Args:
        H: Magnetic field intensity (A/m)
        mu_r: Relative permeability
    
    Returns:
        Flux density B (T)
    """
    MU_0 = 4 * np.pi * 1e-7
    return MU_0 * mu_r * np.asarray(H, dtype=float)

In [3]:
#| export
def frolich(H: np.ndarray, a: float, b: float) -> np.ndarray:
    r"""
    Fröhlich-Kennelly analytical BH approximation.
    
    B = H / (a + b * |H|)
    
    Args:
        H: Magnetic field intensity (A/m)
        a: Model parameter (dimensionless)
        b: Model parameter (m/A)
    
    Returns:
        Flux density B (T)
    """
    H = np.asarray(H, dtype=float)
    return H / (a + b * np.abs(H))

In [4]:
#| export
def fit_frolich(H: np.ndarray, B: np.ndarray) -> tuple[float, float]:
    """
    Fit Fröhlich model parameters (a, b) to measured BH data.
    
    Args:
        H: Measured field intensity (A/m)
        B: Measured flux density (T)
    
    Returns:
        Tuple (a, b) of fitted parameters
    """
    H = np.asarray(H, dtype=float)
    B = np.asarray(B, dtype=float)
    (a, b), _ = curve_fit(frolich, H, B, p0=[1.0, 1.0], maxfev=5000)
    return float(a), float(b)

## Example: Compare BH Models

Let's compute and compare the linear and Fröhlich models.

In [5]:
# Example: Compare linear and Fröhlich models
H_range = np.linspace(0, 5000, 100)
mu_r = 1000

# Linear model
B_linear = linear_region(H_range, mu_r)

# Fröhlich model (example parameters)
a = 0.001
b = 0.0001
B_frolich = frolich(H_range, a, b)

# Display results
print("Sample values from B-H curves:")
print(f"{'H (A/m)':<15} {'B_linear (T)':<20} {'B_Fröhlich (T)':<20}")
print("-" * 55)
for h, bl, bf in zip(H_range[::20], B_linear[::20], B_frolich[::20]):
    print(f"{h:<15.1f} {bl:<20.4f} {bf:<20.4f}")

Sample values from B-H curves:
H (A/m)         B_linear (T)         B_Fröhlich (T)      
-------------------------------------------------------
0.0             0.0000               0.0000              
1010.1          1.2693               9901.9705           
2020.2          2.5387               9950.7438           
3030.3          3.8080               9967.1085           
4040.4          5.0773               9975.3111           


## Tests

In [6]:
# Test linear model
H_test = np.array([0, 100, 1000])
mu_r = 1000
B_test = linear_region(H_test, mu_r)
mu0 = 4 * np.pi * 1e-7
expected = mu0 * mu_r * H_test
assert np.allclose(B_test, expected), "Linear model test failed"
print("✓ linear_region() works correctly")

# Test Fröhlich model
a, b = 0.001, 0.0001
H_frolich = np.array([0, 500, 1000])
B_frolich_vals = frolich(H_frolich, a, b)
assert B_frolich_vals[0] == 0, "Fröhlich should be 0 at H=0"
assert len(B_frolich_vals) == len(H_frolich), "Output length mismatch"
print("✓ frolich() works correctly")

# Test fitting
H_fit = np.array([100, 500, 1000, 2000], dtype=float)
B_fit = np.array([0.6, 1.0, 1.2, 1.3], dtype=float)
a_fit, b_fit = fit_frolich(H_fit, B_fit)
assert isinstance(a_fit, float) and isinstance(b_fit, float)
print(f"✓ fit_frolich() returned a={a_fit:.6f}, b={b_fit:.8f}")
print("\nAll tests passed!")

✓ linear_region() works correctly
✓ frolich() works correctly
✓ fit_frolich() returned a=102.629911, b=0.73616797

All tests passed!
